<a href="https://colab.research.google.com/github/VivekDubey18/LLM-based-trading-system/blob/main/finbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving data.csv to data (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv("data.csv", header=None, encoding="latin-1")

In [ ]:
print(df.head())
print(df.shape)

          0                                                  1
0   neutral  According to Gran , the company has no plans t...
1   neutral  Technopolis plans to develop in stages an area...
2  negative  The international electronic industry company ...
3  positive  With the new production plant the company woul...
4  positive  According to the company 's updated strategy f...
(4846, 2)


In [ ]:
df.columns = ["label", "text"]

In [ ]:
print(df.head())
print(df.shape)

      label                                               text
0   neutral  According to Gran , the company has no plans t...
1   neutral  Technopolis plans to develop in stages an area...
2  negative  The international electronic industry company ...
3  positive  With the new production plant the company woul...
4  positive  According to the company 's updated strategy f...
(4846, 2)


In [ ]:
df["label"] = df["label"].str.strip().str.lower()

In [ ]:
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["label"] = df["label"].map(label_map)

In [ ]:
df = df.dropna()

In [ ]:
print(df.head())
print(df["label"].value_counts())

   label                                               text
0      1  According to Gran , the company has no plans t...
1      1  Technopolis plans to develop in stages an area...
2      0  The international electronic industry company ...
3      2  With the new production plant the company woul...
4      2  According to the company 's updated strategy f...
label
1    2879
2    1363
0     604
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

Step,Training Loss
500,0.647185
1000,0.489453
1500,0.259824
2000,0.234702
2500,0.113646


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2907, training_loss=0.31529431889300746, metrics={'train_runtime': 478.5764, 'train_samples_per_second': 24.297, 'train_steps_per_second': 6.074, 'total_flos': 896332857814800.0, 'train_loss': 0.31529431889300746, 'epoch': 3.0})

In [ ]:
import torch
from sklearn.metrics import accuracy_score, classification_report

predictions = trainer.predict(val_dataset)

preds = torch.argmax(torch.tensor(predictions.predictions), dim=1)

print("Accuracy:", accuracy_score(val_labels, preds))
print(classification_report(val_labels, preds))

Accuracy: 0.8804123711340206
              precision    recall  f1-score   support

           0       0.88      0.89      0.88       110
           1       0.91      0.91      0.91       571
           2       0.83      0.82      0.82       289

    accuracy                           0.88       970
   macro avg       0.87      0.87      0.87       970
weighted avg       0.88      0.88      0.88       970



In [ ]:
trainer.evaluate()

RuntimeError: on_train_begin must be called before on_evaluate

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("./results/checkpoint-2500")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(val_labels, preds))
print(classification_report(val_labels, preds))

Accuracy: 0.8804123711340206
              precision    recall  f1-score   support

           0       0.88      0.89      0.88       110
           1       0.91      0.90      0.91       571
           2       0.82      0.83      0.83       289

    accuracy                           0.88       970
   macro avg       0.87      0.88      0.87       970
weighted avg       0.88      0.88      0.88       970



In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("./results/checkpoint-500")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(val_labels, preds))
print(classification_report(val_labels, preds))

Accuracy: 0.845360824742268
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       110
           1       0.90      0.85      0.88       571
           2       0.75      0.84      0.79       289

    accuracy                           0.85       970
   macro avg       0.83      0.84      0.83       970
weighted avg       0.85      0.85      0.85       970



In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train(resume_from_checkpoint="./results/checkpoint-2500")

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Step,Training Loss
3000,0.110130
3500,0.056249
4000,0.089046
4500,0.015213


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4845, training_loss=0.03284492669828905, metrics={'train_runtime': 377.0586, 'train_samples_per_second': 51.398, 'train_steps_per_second': 12.849, 'total_flos': 1493888096358000.0, 'train_loss': 0.03284492669828905, 'epoch': 5.0})

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("./results/checkpoint-3500")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(val_labels, preds))
print(classification_report(val_labels, preds))

Accuracy: 0.865979381443299
              precision    recall  f1-score   support

           0       0.89      0.86      0.88       110
           1       0.92      0.86      0.89       571
           2       0.77      0.88      0.82       289

    accuracy                           0.87       970
   macro avg       0.86      0.87      0.86       970
weighted avg       0.87      0.87      0.87       970



In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("./results/checkpoint-2000")
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
from transformers import Trainer

trainer = Trainer(
    model=model,
)

import numpy as np
from sklearn.metrics import accuracy_score, classification_report

predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(val_labels, preds))
print(classification_report(val_labels, preds))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy: 0.8731958762886598
              precision    recall  f1-score   support

           0       0.88      0.87      0.88       110
           1       0.90      0.91      0.90       571
           2       0.82      0.80      0.81       289

    accuracy                           0.87       970
   macro avg       0.87      0.86      0.86       970
weighted avg       0.87      0.87      0.87       970



In [ ]:
import shutil

shutil.make_archive("checkpoint_2500", 'zip', "./results/checkpoint-2500")

'/content/checkpoint_2500.zip'

In [ ]:
from google.colab import files

files.download("checkpoint_2500.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>